# Etapa 5 en Colab: fine-tuning de DistilBERT como regresor (GPU)

Corre `src/train_bert.py`, que entrena 4 modelos con el mismo split que los baselines:

| Ejecución | Texto | Pérdida | Para qué |
|---|---|---|---|
| `bert` (main) | OCR limpio | MSE | modelo principal; también predice el texto de referencia (pregunta 2) |
| `bert_cleaning` | OCR crudo / sin baja confianza | MSE | pregunta 3: sensibilidad a la limpieza |
| `bert_huber` | OCR limpio | Huber | pregunta 3: MSE vs Huber |

**Importante: usa una sesión NUEVA de Colab** (no la del notebook de OCR): aquí sí se necesita PyTorch y
PaddlePaddle lo rompe.

- *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU***.
- No hace falta token de Hugging Face: DistilBERT es público y los textos ya están en el repositorio.

In [ ]:
!nvidia-smi

## 1. Repositorio y dependencias

In [ ]:
REPO_URL = 'https://github.com/JOrtega0210/cv_experience_ocr_bert.git'
REPO_DIR = '/content/cv_experience_ocr_bert'
import os
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -3
!ls data/splits data/processed

In [ ]:
# torch ya viene en Colab; se fijan las versiones de requirements.txt para el resto
!pip install -q transformers==5.20.2 accelerate==1.15.0 datasets==5.1.0
import torch, transformers
print(torch.__version__, transformers.__version__, '| GPU:', torch.cuda.get_device_name(0))

## 2. Entrenamiento (≈ 30–45 min en T4)

Hiperparámetros (en `src/train_bert.py`): lr 5e-5, batch 16, hasta 20 épocas con early stopping (paciencia 3)
sobre el MAE de validación, max_length 256, seed 42, fp16.

In [ ]:
!python -m src.train_bert

In [ ]:
import json
for name in ['bert', 'bert_cleaning', 'bert_huber']:
    r = json.load(open(f'output/metrics/{name}.json'))
    for setup, v in r['setups'].items():
        print(f"{name:14s} {setup:17s} best_epoch={v['info']['best_epoch']} train={v['train_seconds']:.0f}s",
              {k: round(m['mae'], 3) for k, m in v['metrics'].items()})

## 3. Guardar resultados en Drive

Genera `results_bert.zip` en `MyDrive/cv_experience_ocr_bert_data/`. Descárgalo a la carpeta de **Descargas**:
contiene predicciones y métricas (`output/`) y el modelo principal (`models/bert/main/`, ~260 MB), que usan la app
y el análisis de errores.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/cv_experience_ocr_bert_data'
os.makedirs(DRIVE_DIR, exist_ok=True)
!rm -f {DRIVE_DIR}/results_bert.zip
!zip -q -r {DRIVE_DIR}/results_bert.zip models/bert/main output/predictions/bert*.csv output/metrics/bert*.json
!ls -lh {DRIVE_DIR}